In [1]:
import os
import shutil
import stat
import glob

candidates = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script/triton/backends/nvidia/bin/ptxas-blackwell")

if not candidates:
    raise FileNotFoundError("ptxas-blackwell not found")

src = candidates[0]

dst_dir = "/tmp/triton-bin"
os.makedirs(dst_dir, exist_ok=True)

dst = f"{dst_dir}/ptxas-blackwell"

shutil.copy2(src, dst)

os.chmod(dst,os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

os.environ["PATH"] = f"{dst_dir}:" + os.environ["PATH"]

os.environ["TRITON_PTXAS_PATH"] = dst
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

os.environ["CUDA_MODULE_LOADING"] = "LAZY"
os.environ["TRITON_DISABLE_AUTOTUNE"] = "1"
os.environ["USE_TRITON"] = "0"

os.environ["PTXAS_PATH"] = dst

print("PTXAS override:", dst)

PTXAS override: /tmp/triton-bin/ptxas-blackwell


In [2]:
#IMPORT MODULES

import sys
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"

site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch

from peft import (LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType)

from transformers import (AutoModelForCausalLM, AutoTokenizer)

import polars as pl
import gc
import math
import re
import random
import numpy as np
import pandas as pd

print("Modules Imported Successfully")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Modules Imported Successfully


In [3]:
print("TRITON_PTXAS_PATH =", os.environ.get("TRITON_PTXAS_PATH"))
print("TRITON_PTXAS_BLACKWELL_PATH =", os.environ.get("TRITON_PTXAS_BLACKWELL_PATH"))
print("PTXAS_PATH =", os.environ.get("PTXAS_PATH"))

"""
Should return:
TRITON_PTXAS_PATH = /tmp/triton-bin/ptxas-blackwell 
TRITON_PTXAS_BLACKWELL_PATH = /tmp/triton-bin/ptxas-blackwell 
PTXAS_PATH = /tmp/triton-bin/ptxas-blackwell
"""

TRITON_PTXAS_PATH = /tmp/triton-bin/ptxas-blackwell
TRITON_PTXAS_BLACKWELL_PATH = /tmp/triton-bin/ptxas-blackwell
PTXAS_PATH = /tmp/triton-bin/ptxas-blackwell


'\nShould return:\nTRITON_PTXAS_PATH = /tmp/triton-bin/ptxas-blackwell \nTRITON_PTXAS_BLACKWELL_PATH = /tmp/triton-bin/ptxas-blackwell \nPTXAS_PATH = /tmp/triton-bin/ptxas-blackwell\n'

In [4]:
#CONFIG

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
MAX_SEQ_LEN = 1024
SEED = 42

In [5]:
#LOAD DATASET

TRAIN_CSV = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"

train = pl.read_csv(TRAIN_CSV)
train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [6]:
#LOAD MODEL

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16)

print("Model Loaded Successfully")

print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model Loaded Successfully
Initializing LoRA adapter with rank=32...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [7]:
import subprocess

subprocess.check_output([os.environ["TRITON_PTXAS_BLACKWELL_PATH"], "--version"]).decode()

"""
Should return 'ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Tue_May_27_02:18:05_PDT_2025\nCuda compilation tools, release 12.9, V12.9.86\nBuild cuda_12.9.r12.9/compiler.36037853_0\n'
"""

"\nShould return 'ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Tue_May_27_02:18:05_PDT_2025\nCuda compilation tools, release 12.9, V12.9.86\nBuild cuda_12.9.r12.9/compiler.36037853_0\n'\n"

In [8]:
#YOUR TRAINING CODE BLOCK HERE

In [9]:
#SAVE ADAPTER
print(f"Saving adapter to {OUTPUT_DIR}...", flush=True)
model.save_pretrained(OUTPUT_DIR)

import subprocess
subprocess.run("zip -m submission.zip *", shell=True, check=True)
print("Adapter Saved Successfully")

Saving adapter to /kaggle/working...
  adding: README.md (deflated 66%)
  adding: __notebook__.ipynb (deflated 77%)
  adding: adapter_config.json (deflated 55%)
  adding: adapter_model.safetensors (deflated 77%)
Adapter Saved Successfully
